In [ ]:
# Configuration : rend le dossier src/ importable et récupère les chemins du projet
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import RAW_DIR, PROCESSED_DIR, FIGURES_DIR
print("Données brutes :", RAW_DIR)
print("Données traitées :", PROCESSED_DIR)

In [1]:
import pandas as pd

df = pd.read_csv(RAW_DIR / "#20.csv", nrows=20)

print(df.to_string(index=False))

 Unnamed: 0    record_time  soc  pack_voltage (V)  charge_current (A)  max_cell_voltage (V)  min_cell_voltage (V)  max_temperature (℃)  min_temperature (℃)  available_energy (kw)  available_capacity (Ah)
          0 20190726110304 50.0             336.1           -45.00000                 3.748                 3.728                   31                   30                  22.61                    67.96
          1 20190726110312 50.0             336.4           -45.00000                 3.751                 3.730                   31                   30                  22.64                    68.06
          2 20190726110320 50.0             336.8           -45.00000                 3.754                 3.733                   31                   30                  22.67                    68.16
          3 20190726110328 50.0             336.9           -45.00000                 3.755                 3.734                   31                   30                  22.70      

In [2]:
import pandas as pd
from pathlib import Path

fichiers = list(RAW_DIR.glob("*.csv"))
colonnes_reference = list(pd.read_csv(fichiers[0], nrows=0).columns)

for fichier in fichiers[1:]:
    colonnes = list(pd.read_csv(fichier, nrows=0).columns)

    if colonnes != colonnes_reference:
        print(f"Différence détectée : {fichier.name}")
        print("Colonnes manquantes :", set(colonnes_reference) - set(colonnes))
        print("Colonnes supplémentaires :", set(colonnes) - set(colonnes_reference))

print("Vérification terminée.")
print(f"Nombre de fichiers analysés : {len(fichiers)}")
print(f"Nombre de colonnes de référence : {len(colonnes_reference)}")

Différence détectée : dataset_batterie.csv
Colonnes manquantes : {'Unnamed: 0'}
Colonnes supplémentaires : {'vehicle_id'}
Vérification terminée.
Nombre de fichiers analysés : 21
Nombre de colonnes de référence : 11


In [3]:
df.shape

(20, 11)

In [4]:
import pandas as pd
from pathlib import Path

fichiers = sorted(RAW_DIR.glob("#*.csv"))

df = pd.concat(
    [pd.read_csv(f) for f in fichiers],
    ignore_index=True
)

df.to_csv(PROCESSED_DIR / "dataset_batterie.csv", index=False)

print("Fusion terminée !")
print("Dimensions :", df.shape)

Fusion terminée !
Dimensions : (16100728, 11)


In [5]:
df.shape

(16100728, 11)

In [6]:
import pandas as pd
from pathlib import Path

fichier = PROCESSED_DIR / "dataset_batterie.csv"

df = pd.read_csv(fichier, nrows=1000)

print("Dimensions échantillon :", df.shape)
print("\nValeurs manquantes :")
print(df.isnull().sum())

print("\nStatistiques descriptives :")
print(df.describe().round(2))

Dimensions échantillon : (1000, 11)

Valeurs manquantes :
Unnamed: 0                 0
record_time                0
soc                        0
pack_voltage (V)           0
charge_current (A)         0
max_cell_voltage (V)       0
min_cell_voltage (V)       0
max_temperature (℃)        0
min_temperature (℃)        0
available_energy (kw)      0
available_capacity (Ah)    0
dtype: int64

Statistiques descriptives :
       Unnamed: 0   record_time      soc  pack_voltage (V)  \
count     1000.00  1.000000e+03  1000.00           1000.00   
mean       499.50  2.019073e+13    64.54            352.31   
std        288.82  2.207832e+05    23.15             17.91   
min          0.00  2.019073e+13    27.20            328.20   
25%        249.75  2.019073e+13    42.40            334.10   
50%        499.50  2.019073e+13    64.40            349.80   
75%        749.25  2.019073e+13    86.00            369.40   
max        999.00  2.019073e+13    99.60            381.00   

       charge_current 

In [7]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet(PROCESSED_DIR / "dataset_batterie.parquet")

# ---------- 1. NETTOYAGE (erreurs capteur) ----------
avant = len(df)
df = df.drop_duplicates(subset=["vehicle_id", "time"])                 # doublons d'horodatage
df.loc[df["cell_v_min"] <= 0, "cell_v_min"] = pd.NA                     # tension 0 V = capteur
df.loc[~df["temp_max"].between(-30, 80), "temp_max"] = pd.NA           # température impossible
print(f"Doublons supprimés : {avant - len(df):,}")

# ---------- 2. INDICATEURS ----------
df["ecart_tension"] = df["cell_v_max"] - df["cell_v_min"]

print("\n=== Distribution réelle (pour ajuster les seuils) ===")
print(df[["temp_max", "ecart_tension"]].quantile([0.5, 0.9, 0.99, 0.999]).round(3))

# ---------- 3. DÉTECTION ----------
df["surchauffe_alerte"]   = df["temp_max"] > 45
df["surchauffe_critique"] = df["temp_max"] > 50
df["cellule_defaillante"] = df["ecart_tension"] > 0.1

# anomalie confirmée = au moins 3 mesures consécutives (évite le bruit ponctuel)
def confirme(col):
    return (df.groupby("vehicle_id")[col]
              .transform(lambda s: s.astype(int).rolling(3).sum() == 3))

df["surchauffe_confirmee"] = confirme("surchauffe_alerte")
df["cellule_confirmee"]    = confirme("cellule_defaillante")

# ---------- 4. RÉSUMÉ PAR VÉHICULE ----------
resume = df.groupby("vehicle_id").agg(
    mesures=("time", "size"),
    temp_max=("temp_max", "max"),
    surchauffe_45=("surchauffe_alerte", "sum"),
    surchauffe_50=("surchauffe_critique", "sum"),
    surchauffe_conf=("surchauffe_confirmee", "sum"),
    ecart_v_max=("ecart_tension", "max"),
    ecart_v_moyen=("ecart_tension", "mean"),
    cellule_0_1V=("cellule_defaillante", "sum"),
    cellule_conf=("cellule_confirmee", "sum"),
)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
print("\n=== Anomalies par véhicule ===")
print(resume.round(3))

df.to_parquet(PROCESSED_DIR / "dataset_anomalies.parquet", index=False)
print("\nSauvegardé : dataset_anomalies.parquet")

,Unnamed: 0,record_time,soc,pack_voltage (V),charge_current (A),max_cell_voltage (V),min_cell_voltage (V),max_temperature (℃),min_temperature (℃),available_energy (kw),available_capacity (Ah)
0,0,20190726200235,27.2,328.2,-52.20001,3.656,3.640,41,38,12.40,37.28
1,1,20190726200243,27.6,328.5,-52.20001,3.663,3.645,41,38,12.44,37.39
2,2,20190726200251,27.6,328.6,-52.20001,3.665,3.647,41,38,12.47,37.51
3,3,20190726200259,27.6,328.6,-52.20001,3.666,3.649,41,38,12.52,37.64
4,4,20190726200307,27.6,328.8,-52.20001,3.666,3.649,41,38,12.56,37.76
5,5,20190726200315,28.0,328.8,-52.20001,3.669,3.650,41,38,12.59,37.87
6,6,20190726200323,28.0,328.9,-52.20001,3.669,3.651,41,38,12.64,37.99
7,7,20190726200331,28.0,328.9,-52.20001,3.669,3.651,41,38,12.67,38.11
8,8,20190726200339,28.0,329.1,-52.20001,3.670,3.653,41,38,12.71,38.20
9,9,20190726200347,28.0,329.1,-52.20001,3.671,3.652,41,38,12.75,38.34


In [8]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Unnamed: 0               1000 non-null   int64  
 1   record_time              1000 non-null   int64  
 2   soc                      1000 non-null   float64
 3   pack_voltage (V)         1000 non-null   float64
 4   charge_current (A)       1000 non-null   float64
 5   max_cell_voltage (V)     1000 non-null   float64
 6   min_cell_voltage (V)     1000 non-null   float64
 7   max_temperature (℃)      1000 non-null   int64  
 8   min_temperature (℃)      1000 non-null   int64  
 9   available_energy (kw)    1000 non-null   float64
 10  available_capacity (Ah)  1000 non-null   float64
dtypes: float64(7), int64(4)
memory usage: 86.1 KB


In [9]:
missing_percent = df.isnull().mean() * 100
print(missing_percent)


Unnamed: 0                 0.0
record_time                0.0
soc                        0.0
pack_voltage (V)           0.0
charge_current (A)         0.0
max_cell_voltage (V)       0.0
min_cell_voltage (V)       0.0
max_temperature (℃)        0.0
min_temperature (℃)        0.0
available_energy (kw)      0.0
available_capacity (Ah)    0.0
dtype: float64


In [10]:
df.dtypes

Unnamed: 0                   int64
record_time                  int64
soc                        float64
pack_voltage (V)           float64
charge_current (A)         float64
max_cell_voltage (V)       float64
min_cell_voltage (V)       float64
max_temperature (℃)          int64
min_temperature (℃)          int64
available_energy (kw)      float64
available_capacity (Ah)    float64
dtype: object

In [11]:
import pandas as pd
from pathlib import Path

# tri numérique : #1, #2, ..., #10 (et non #1, #10, #11, #2)
fichiers = sorted(RAW_DIR.glob("#*.csv"), key=lambda f: int(f.stem.lstrip("#")))

frames = []
for f in fichiers:
    d = pd.read_csv(f, index_col=0)             # supprime "Unnamed: 0"
    d.insert(0, "vehicle_id", int(f.stem.lstrip("#")))
    frames.append(d)
    print(f"{f.name}: {len(d):,} lignes")

df = pd.concat(frames, ignore_index=True)
df["record_time"] = pd.to_datetime(df["record_time"])
df = df.sort_values(["vehicle_id", "record_time"]).reset_index(drop=True)

df.to_csv(PROCESSED_DIR / "dataset_batterie.csv", index=False)
print("\nFusion terminée :", df.shape, "| véhicules :", df["vehicle_id"].nunique())

#1.csv: 854,591 lignes
#2.csv: 841,810 lignes
#3.csv: 802,918 lignes
#4.csv: 844,537 lignes
#5.csv: 832,189 lignes
#6.csv: 815,202 lignes
#7.csv: 808,770 lignes
#8.csv: 798,496 lignes
#9.csv: 792,060 lignes
#10.csv: 728,849 lignes
#11.csv: 809,470 lignes
#12.csv: 825,533 lignes
#13.csv: 818,352 lignes
#14.csv: 793,132 lignes
#15.csv: 704,958 lignes
#16.csv: 819,567 lignes
#17.csv: 767,868 lignes
#18.csv: 856,192 lignes
#19.csv: 778,745 lignes
#20.csv: 807,489 lignes

Fusion terminée : (16100728, 11) | véhicules : 20


In [12]:
df.to_parquet(PROCESSED_DIR / "dataset_batterie.parquet", index=False)


In [13]:
import pandas as pd
from pathlib import Path

# ---------- 1. FUSION avec vehicle_id et dates corrigées ----------
fichiers = sorted(RAW_DIR.glob("#*.csv"), key=lambda f: int(f.stem.lstrip("#")))
frames = []
for f in fichiers:
    d = pd.read_csv(f, index_col=0)
    d.insert(0, "vehicle_id", int(f.stem.lstrip("#")))
    frames.append(d)
df = pd.concat(frames, ignore_index=True)

df.columns = ["vehicle_id", "time", "soc", "pack_v", "current", "cell_v_max", "cell_v_min",
              "temp_max", "temp_min", "energy", "capacity"]

df["time"] = pd.to_datetime(df["time"].astype(str), format="%Y%m%d%H%M%S")
df = df.sort_values(["vehicle_id", "time"]).reset_index(drop=True)
df.to_parquet(PROCESSED_DIR / "dataset_batterie.parquet", index=False)
print("Fusion OK :", df.shape)

# ---------- 2. RAPPORT QUALITÉ par véhicule ----------
df["dt_s"] = df.groupby("vehicle_id")["time"].diff().dt.total_seconds()
g = df.groupby("vehicle_id")

rapport = pd.DataFrame({
    "debut": g["time"].min(),
    "fin": g["time"].max(),
    "pas_median_s": g["dt_s"].median(),
    "trous_10min": g["dt_s"].apply(lambda s: (s > 600).sum()),
    "doublons": g["dt_s"].apply(lambda s: (s == 0).sum()),
    "cell_v_0": g["cell_v_min"].apply(lambda s: (s == 0).sum()),
    "courant_-400": g["current"].apply(lambda s: (s <= -400).sum()),
    "manquants": df.drop(columns="dt_s").isna().groupby(df["vehicle_id"]).sum().sum(axis=1),
})
rapport["duree_jours"] = (rapport["fin"] - rapport["debut"]).dt.days

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
print("\n=== RAPPORT PAR VÉHICULE ===")
print(rapport)

print("\n=== DESCRIBE GLOBAL ===")
print(df.drop(columns=["vehicle_id", "time", "dt_s"]).describe().round(3).T)

Fusion OK : (16100728, 11)

=== RAPPORT PAR VÉHICULE ===
                         debut                 fin  pas_median_s  trous_10min  doublons  cell_v_0  courant_-400  manquants  duree_jours
vehicle_id                                                                                                                             
1          2019-07-26 20:02:35 2021-11-15 16:46:40           8.0         1619        24         0             0          0          842
2          2019-07-25 15:14:26 2021-11-15 14:12:56           8.0         1649      2290         1             0          0          843
3          2019-07-26 19:08:05 2021-11-15 22:20:40           8.0         1543      8467         2             0          0          843
4          2019-07-25 15:58:38 2021-11-15 17:57:51           8.0         1623      2538         0             0          0          844
5          2019-07-26 00:22:51 2021-11-15 17:44:04           8.0         1610      4610         1             0          0     

In [14]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet(PROCESSED_DIR / "dataset_batterie.parquet")

pd.set_option("display.max_columns", None)   # affiche toutes les colonnes
pd.set_option("display.width", 250)          # évite le retour à la ligne

print(df.head(10))     # 10 premières lignes

   vehicle_id                time   soc  pack_v   current  cell_v_max  cell_v_min  temp_max  temp_min  energy  capacity
0           1 2019-07-26 20:02:35  27.2   328.2 -52.20001       3.656       3.640        41        38   12.40     37.28
1           1 2019-07-26 20:02:43  27.6   328.5 -52.20001       3.663       3.645        41        38   12.44     37.39
2           1 2019-07-26 20:02:51  27.6   328.6 -52.20001       3.665       3.647        41        38   12.47     37.51
3           1 2019-07-26 20:02:59  27.6   328.6 -52.20001       3.666       3.649        41        38   12.52     37.64
4           1 2019-07-26 20:03:07  27.6   328.8 -52.20001       3.666       3.649        41        38   12.56     37.76
5           1 2019-07-26 20:03:15  28.0   328.8 -52.20001       3.669       3.650        41        38   12.59     37.87
6           1 2019-07-26 20:03:23  28.0   328.9 -52.20001       3.669       3.651        41        38   12.64     37.99
7           1 2019-07-26 20:03:31  28.0 

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16100728 entries, 0 to 16100727
Data columns (total 11 columns):
 #   Column      Dtype         
---  ------      -----         
 0   vehicle_id  int64         
 1   time        datetime64[ns]
 2   soc         float64       
 3   pack_v      float64       
 4   current     float64       
 5   cell_v_max  float64       
 6   cell_v_min  float64       
 7   temp_max    int64         
 8   temp_min    int64         
 9   energy      float64       
 10  capacity    float64       
dtypes: datetime64[ns](1), float64(7), int64(3)
memory usage: 1.3 GB


In [18]:
df.describe().round(1)

,vehicle_id,time,soc,pack_v,current,cell_v_max,cell_v_min,temp_max,temp_min,energy,capacity
count,16100728.0,16100728,16100728.0,16100728.0,16100728.0,16100728.0,16100728.0,16100728.0,16100728.0,16100728.0,16100728.0
mean,10.4,2020-08-16 03:09:24.495682816,64.0,352.1,-53.0,3.9,3.9,33.9,32.0,27.7,83.3
min,1.0,2019-07-22 16:35:17,0.0,312.8,-400.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,5.0,2019-12-21 03:01:08.249999872,47.6,337.1,-70.0,3.8,3.7,30.0,28.0,20.6,62.0
50%,10.0,2020-08-08 22:34:38,66.0,351.9,-50.2,3.9,3.9,34.0,33.0,28.5,85.7
75%,16.0,2021-03-20 12:05:59.500000,82.4,366.1,-43.9,4.1,4.1,38.0,36.0,35.6,106.9
max,20.0,2021-11-16 01:05:42,100.0,386.2,12.1,4.3,4.2,49.0,48.0,45.3,136.2
std,5.8,NaN,22.0,16.2,17.3,0.2,0.2,5.6,5.6,9.6,28.9


In [19]:
df.shape

(16100728, 11)